# 04 — Statistical Mock Prediction (Monthly + Weekly)

This notebook generates future water quality values using a statistical mock
forecasting approach. Monthly forecasts are produced as the primary outputs
based on satellite-derived statistics, while weekly values are optionally
generated via temporal downscaling for visualization and early-warning purposes.


Cell 2 — Import Libraries

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


Cell 3 — Paths & Settings

In [2]:
PREP_DIR = "../data/prepared"
STATS_DIR = "../data/stats"
OUT_DIR = "../outputs/forecasts"
FIG_DIR = "../outputs/figures"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# =========================
# CONFIG (ปรับได้ตรงนี้)
# =========================
RESOLUTION = "monthly"   # "monthly" | "weekly"
HORIZON = 12             # monthly = เดือน, weekly = สัปดาห์

PARAMETERS = [
    "secchi",
    "chlorophyll_a",
    "tsi",
    "turbidity",
    "salinity",
    "do",
    "ph"
]


Cell 4 — Physical Bounds (Clamp)

In [3]:
PHYSICAL_BOUNDS = {
    "secchi": (0, None),
    "chlorophyll_a": (0, None),
    "tsi": (30, 80),
    "turbidity": (0, None),
    "salinity": (0, None),
    "do": (0, None),
    "ph": (0, 14)
}


Cell 5 — Noise Factor (ควบคุมความสมจริง)

In [4]:
NOISE_FACTOR = {
    "secchi": 0.3,
    "chlorophyll_a": 0.3,
    "tsi": 0.25,
    "turbidity": 0.35,
    "salinity": 0.3,
    "do": 0.1,     # DO ต้องนิ่ง
    "ph": 0.2
}


Cell 6 — Helper: Clamp

In [5]:
def clamp(value, low=None, high=None):
    if low is not None:
        value = max(value, low)
    if high is not None:
        value = min(value, high)
    return value


Cell 7 — Helper: Scale Monthly → Weekly (Downscaling)

In [6]:
def scale_monthly_to_weekly(stats):
    """
    Temporal downscaling from monthly to weekly:
    - trend_w = trend_m / 4
    - std_w   = std_m / sqrt(4)
    """
    return {
        "mean": stats["mean"],
        "trend": stats["trend"] / 4.0,
        "std": stats["std"] / np.sqrt(4.0)
    }


Cell 8 — Helper: Forecast Series (Generic)

In [7]:
def forecast_series(last_value, stats, param, horizon):
    mean = stats["mean"]
    std = stats["std"]
    trend = stats["trend"]

    low, high = PHYSICAL_BOUNDS.get(param, (None, None))
    noise_scale = NOISE_FACTOR.get(param, 0.3)

    values = []
    current = last_value

    for _ in range(horizon):
        noise = np.random.normal(0, std * noise_scale)
        next_val = current + trend + noise
        next_val = clamp(next_val, low, high)
        values.append(next_val)
        current = next_val

    return values


Cell 9 — MAIN LOOP: Forecast ALL Stations

In [8]:
for file in os.listdir(PREP_DIR):
    if not file.endswith("_prepared.csv"):
        continue

    station = file.replace("_prepared.csv", "")
    print(f"🚀 Forecasting: {station} ({RESOLUTION})")

    # ---------- Load prepared data ----------
    df = pd.read_csv(
        os.path.join(PREP_DIR, file),
        parse_dates=["date"],
        index_col="date"
    )

    # ---------- Load stats ----------
    with open(os.path.join(STATS_DIR, f"{station}_stats.json")) as f:
        stats_all = json.load(f)

    # ---------- Date range ----------
    last_date = df.index.max()
    if RESOLUTION == "monthly":
        future_dates = pd.date_range(
            last_date + pd.offsets.MonthEnd(1),
            periods=HORIZON,
            freq="ME"
        )
    else:  # weekly
        future_dates = pd.date_range(
            last_date + pd.offsets.Week(1),
            periods=HORIZON,
            freq="W"
        )

    forecast_df = pd.DataFrame(index=future_dates)

    # ---------- Forecast each parameter ----------
    for param in PARAMETERS:
        if param not in df.columns or param not in stats_all:
            continue

        last_value = df[param].dropna().iloc[-1]
        stats_param = stats_all[param]

        # Downscale if weekly
        if RESOLUTION == "weekly":
            stats_param = scale_monthly_to_weekly(stats_param)

        forecast_df[param] = forecast_series(
            last_value, stats_param, param, HORIZON
        )

    # ---------- Save CSV ----------
    out_csv = os.path.join(
        OUT_DIR, f"{station}_{RESOLUTION}_forecast.csv"
    )
    forecast_df.to_csv(out_csv)

    # ---------- Plot (Secchi example) ----------
    if "secchi" in df.columns and "secchi" in forecast_df.columns:
        plt.figure(figsize=(10,4))
        plt.plot(df.index, df["secchi"], label="Historical")
        plt.plot(
            forecast_df.index, forecast_df["secchi"],
            linestyle="--", label=f"Mock Forecast ({RESOLUTION})"
        )
        plt.title(f"{station} — Secchi Depth ({RESOLUTION.capitalize()} Forecast)")
        plt.legend()
        plt.tight_layout()

        fig_path = os.path.join(
            FIG_DIR, f"{station}_secchi_{RESOLUTION}_forecast.png"
        )
        plt.savefig(fig_path)
        plt.close()

    print(f"✅ Saved: {station}_{RESOLUTION}")


🚀 Forecasting: CP01 (monthly)
✅ Saved: CP01_monthly
🚀 Forecasting: LS01 (monthly)
✅ Saved: LS01_monthly
🚀 Forecasting: LS03 (monthly)
✅ Saved: LS03_monthly
🚀 Forecasting: PN01 (monthly)
✅ Saved: PN01_monthly
🚀 Forecasting: SK01 (monthly)
✅ Saved: SK01_monthly
🚀 Forecasting: SK06 (monthly)
✅ Saved: SK06_monthly
🚀 Forecasting: TP011 (monthly)
✅ Saved: TP011_monthly
🚀 Forecasting: TP01 (monthly)
✅ Saved: TP01_monthly
🚀 Forecasting: TP04 (monthly)
✅ Saved: TP04_monthly


Cell 10 — Inspect One Output

In [10]:
sample = pd.read_csv(
    f"../outputs/forecasts/CP01_{RESOLUTION}_forecast.csv",
    parse_dates=[0],
    index_col=0
)

sample.head()


,secchi,chlorophyll_a,tsi,turbidity,salinity,do,ph
2026-01-31,1.487279,12.581087,55.413773,109.649583,0.014455,9.577086,7.434211
2026-02-28,1.571819,11.770845,54.762889,115.421778,0.016370,9.577073,7.289748
2026-03-31,1.550956,11.433371,54.801466,115.744379,0.014431,9.577037,6.937464
2026-04-30,1.465124,11.461243,54.552251,115.225500,0.008398,9.577065,7.050556
2026-05-31,1.476069,10.399506,55.353136,121.346491,0.011148,9.577068,7.090490
